# Reconocimiento de figuras musicales con red convolucional densa

Este proyecto implementa el entrenamiento, evaluación métrica e inferencia de un modelo convolucional denso para la clasificación de símbolos musicales (figuras rítmicas, armaduras de tonalidad, barras de unión compuestas, silencios y claves) a partir de imágenes monocromáticas de trazo.

### Información general del proyecto

- **Curso:** Computación Gráfica (CC431A)
- **Institución:** Universidad Nacional de Ingeniería — Facultad de Ciencias
- **Integrantes:**
  - Anngie Salazar Álvarez (Código: `20162195F`)
  - Gabriel Arturo Rincón Miranda (Código: `20231285E`)
  - Erick Daniel Ortega Morán (Código: `20210209H`)
- **Demostración pública:** [https://clasificador-de-simbolos-musicales.onrender.com](https://clasificador-de-simbolos-musicales.onrender.com)

> **Nota operativa del host:** Si el servicio ha estado inactivo durante más de 15 minutos, la instancia entra en suspensión temporal automática en el plan gratuito de Render; la primera solicitud puede tardar entre 30 y 50 segundos mientras se reactiva la máquina virtual. Asimismo, el análisis por la red neuronal puede requerir unos instantes adicionales debido a los recursos acotados asignados por el host (0.1 CPU, 512 MB RAM).

### Fundamentación teórica de la arquitectura DenseNet

A diferencia de las redes convolucionales tradicionales donde la capa $l$ recibe únicamente la salida de la capa previa $x_{l-1}$, una red densa (*Densely Connected Convolutional Network*) conecta directamente cada capa con todas las subsiguientes mediante concatenación de características:

$$x_l = H_l([x_0, x_1, \dots, x_{l-1}])$$

donde:
- $[x_0, x_1, \dots, x_{l-1}]$ denota la concatenación de los mapas de características generados en todas las capas anteriores a lo largo de la dimensión de canales.
- $H_l(\cdot)$ es una transformación no lineal compuesta definida por la secuencia: **Normalización por lotes (BatchNorm) $\to$ Activación no lineal (ReLU) $\to$ Convolución bidimensional ($3 \times 3$)**.
- Tasa de crecimiento ($k=16$): cada función $H_l$ produce $k$ nuevos mapas de canales. Si el volumen inicial posee $k_0$ canales, la capa $l$ recibe un tensor con $k_0 + l \cdot k$ canales.

### Capas de transición y compresión

Para controlar el crecimiento de la dimensionalidad entre bloques densos consecutivos, se intercala una capa de transición compuesta por:
1. Convolución $1 \times 1$ de compresión con factor $\theta = 0.5$, reduciendo a la mitad la cantidad de canales acumulados: $m = \lfloor \theta \cdot m_{\text{in}} \rfloor$.
2. Reducción espacial mediante submuestreo de promedio (*Average Pooling*) $2 \times 2$ con paso (*stride*) de 2.

### Agrupamiento global y función de costo

Al finalizar las etapas convolucionales, se aplica un agrupamiento promedio global (*Global Average Pooling*, GAP) que promedia cada mapa de características en una sola magnitud escalar ($C \times H \times W \to C \times 1 \times 1$). Esto desacopla el clasificador lineal de la resolución espacial, previene el sobreajuste y minimiza el número de parámetros libres.

La optimización se realiza mediante la función de pérdida de entropía cruzada multiclase:

$$\mathcal{L}_{\text{CE}} = - \sum_{c=1}^C y_c \log \left( \frac{\exp(z_c)}{\sum_{j=1}^C \exp(z_j)} \right)$$

donde $y_c \in \{0, 1\}$ es el vector unitario que codifica la clase real y $z_c$ representa el logit generado por la capa lineal final para la clase $c$.

In [ ]:
import os
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
import torch

base_dir = os.path.abspath('.')
if base_dir not in sys.path:
    sys.path.insert(0, base_dir)

from src.model import DenseNetMusical
from src.dataset import create_dataloaders, load_classes_metadata
from src.predict import MusicalSymbolPredictor

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Dispositivo de cómputo en uso:', device)

## Metodología de recolección y curación de datos

Para lograr robustez ante trazos interactivos en pantalla, el conjunto de entrenamiento integra una base de datos híbrida:

1. **Recolección de dibujos propios en lienzo interactivo:** A través del módulo `recolector.py`, el equipo capturó **1,200 dibujos manuales** (150 por categoría) trazados con mouse en un lienzo de 220×220 px y grosor de 12 px (idéntico al entorno de despliegue). El recolector balancea automáticamente las clases solicitando siempre el símbolo con menor conteo.
2. **Base de referencia pública:** Se seleccionaron muestras complementarias del repositorio Rebelo (`db1` y `db2`) para completar 650 instancias por clase.
3. **Fusión y balanceo estricto:** 150 dibujos propios + 650 públicos = **800 muestras por categoría (6,400 en total)**, eliminando cualquier sesgo hacia clases mayoritarias.
4. **Preprocesamiento morfológico:** Binarización invertida (fondo negro, trazo blanco), centrado de masa con margen perimetral del 15% e interpolación bilineal a resolución de $64 \times 64$ píxeles.
5. **Partición estratificada desacoplada:**
   - 70% entrenamiento (4,480 muestras).
   - 10% validación (640 muestras).
   - 20% prueba independiente (1,280 muestras).

In [ ]:
classes = load_classes_metadata()
data_path = os.path.join(base_dir, 'dataset', 'data_musical.npz')
data = np.load(data_path)
X, y = data['X'], data['y']

print(f'Total de muestras procesadas: {len(X)}')
print(f'Dimensiones del tensor: {X.shape[1:]} (resolución 64x64, 1 canal)')
print('\nDistribución balanceada por categoría:')
for c in classes:
    count = int(np.sum(y == c['id']))
    print(f"  [{c['id']}] {c['symbol']} {c['name']:28s} | Muestras: {count} ({c['category']})")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
axes = axes.flatten()
for i, c in enumerate(classes):
    idx = np.where(y == c['id'])[0][10]
    axes[i].imshow(X[idx], cmap='gray')
    axes[i].set_title(c['name'], fontsize=9)
    axes[i].axis('off')
plt.tight_layout()
plt.show()

## Arquitectura del modelo `DenseNetMusical` y especificaciones de componentes

### Manual de especificaciones interno de componentes

1. **`ConvBlock(in_channels, growth_rate)`:**
   - **Atributos:** `bn` (`nn.BatchNorm2d`), `relu` (`nn.ReLU`), `conv` (`nn.Conv2d` con filtro $3 \times 3$, relleno 1, sin bias).
   - **Método `forward(x)`:** Aplica $H_l(x) = \text{conv}(\text{relu}(\text{bn}(x)))$ y concatena la entrada: `torch.cat([x, out], dim=1)`.

2. **`DenseBlock(in_channels, num_layers, growth_rate)`:**
   - **Atributos:** `layers` (`nn.ModuleList` de `ConvBlock`), `out_channels` ($c_{\text{in}} + L \cdot k$).
   - **Método `forward(x)`:** Encadena las transformaciones densas expandiendo progresivamente la dimensión de canales.

3. **`TransitionLayer(in_channels, out_channels)`:**
   - **Atributos:** `bn`, `relu`, `conv` (filtro $1 \times 1$, factor $\theta = 0.5$), `pool` (`AvgPool2d` $2 \times 2$, stride 2).
   - **Método `forward(x)`:** Comprime los canales a la mitad y reduce la resolución espacial espacial a la mitad.

4. **`DenseNetMusical(in_channels=1, num_classes=8, growth_rate=16, block_config=(3,3,3))`:**
   - **Atributos:** `init_conv` (32 canales, $64 \times 64$), `features` (3 bloques densos y 2 transiciones), `gap` (`AdaptiveAvgPool2d(1, 1)`), `classifier` (`nn.Linear` 92 a 8).
   - **Métodos principales:**
     - `extract_features(x)`: Extrae mapas convolucionales hasta $92 \times 16 \times 16$.
     - `forward(x)`: Extrae características, aplica GAP y computa los logits lineales.
     - `get_intermediate_activations(x)`: Extrae diccionarios con representaciones de la convolución inicial, primera transición y mapas profundos.
     - `compute_cam(x, class_idx)`: Proyecta los pesos del clasificador para calcular mapas de activación de clase.

In [ ]:
model = DenseNetMusical(in_channels=1, num_classes=len(classes), growth_rate=16).to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parámetros totales: {total_params:,}')
print(f'Parámetros entrenables: {trainable_params:,}')

## Curvas de convergencia y optimización

El modelo se entrenó durante 15 épocas mediante el algoritmo Adam ($\eta_0 = 0.001$), incorporando el programador adaptativo `ReduceLROnPlateau` que reduce la tasa de aprendizaje a la mitad ante estancamientos en la pérdida de validación.

In [ ]:
train_loader, val_loader, test_loader, _ = create_dataloaders(data_file_path=data_path, batch_size=32)

hist_path = os.path.join(base_dir, 'models', 'training_history.json')
with open(hist_path, 'r', encoding='utf-8') as f:
    hist = json.load(f)

epochs = range(1, len(hist['train_loss']) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.plot(epochs, hist['train_loss'], label='Entrenamiento', color='#2563eb', linewidth=1.8)
ax1.plot(epochs, hist['val_loss'], label='Validación', color='#dc2626', linewidth=1.8)
ax1.set_title('Pérdida (Entropía cruzada)', fontsize=10)
ax1.set_xlabel('Época')
ax1.legend()
ax1.grid(True, linestyle='--', alpha=0.5)

ax2.plot(epochs, [a * 100 for a in hist['train_acc']], label='Entrenamiento', color='#2563eb', linewidth=1.8)
ax2.plot(epochs, [a * 100 for a in hist['val_acc']], label='Validación', color='#dc2626', linewidth=1.8)
ax2.set_title('Exactitud (%)', fontsize=10)
ax2.set_xlabel('Época')
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## Evaluación cuantitativa y dominio de las métricas

A continuación se describe el dominio, definición matemática e interpretación técnica de cada dimensión evaluada:

1. **Exactitud Top-1 ($[0, 100]\%$):**
   $$\text{Top-1} = \frac{1}{N} \sum_{i=1}^N \mathbb{I}(\hat{y}_{i, 1} = y_i)$$
   Representa el porcentaje de muestras donde la categoría de mayor probabilidad coincide exactamente con la etiqueta real.

2. **Exactitud Top-2 ($[0, 100]\%$):**
   $$\text{Top-2} = \frac{1}{N} \sum_{i=1}^N \mathbb{I}(y_i \in \{\hat{y}_{i, 1}, \hat{y}_{i, 2}\})$$
   Cuantifica si la etiqueta verdadera se encuentra entre las 2 primeras opciones más verosímiles del modelo.

3. **Precisión ($[0, 100]\%$):**
   $$\text{Precisión} = \frac{TP}{TP + FP}$$
   Mide la fracción de predicciones de una clase que resultaron verdaderas.

4. **Exhaustividad / Recall ($[0, 100]\%$):**
   $$\text{Recall} = \frac{TP}{TP + FN}$$
   Mide la proporción de muestras reales de una categoría identificadas con éxito por el modelo.

5. **F1-Score ($[0, 100]\%$):**
   $$\text{F1} = 2 \cdot \frac{\text{Precisión} \cdot \text{Recall}}{\text{Precisión} + \text{Recall}}$$
   Media armónica entre precisión y exhaustividad.

6. **Latencia de inferencia y rendimiento ($[0, \infty)$ ms, fps):**
   Tiempo promedio de procesamiento hacia adelante por muestra en procesador CPU.

7. **Margen de certidumbre ($\Delta P \in [0, 100]\%$):**
   $$\Delta P = \bar{P}_{\text{aciertos}} - \bar{P}_{\text{errores}}$$
   Diferencia promedio entre la confianza softmax en predicciones correctas frente a incorrectas.

In [ ]:
metrics_path = os.path.join(base_dir, 'models', 'test_metrics.json')
with open(metrics_path, 'r', encoding='utf-8') as f:
    metrics = json.load(f)

print(f"Exactitud Top-1 en conjunto de prueba: {metrics['top1_accuracy']}%")
print(f"Exactitud Top-2 en conjunto de prueba: {metrics['top2_accuracy']}%")
print(f"Latencia media en CPU: {metrics['latencia_media_ms']} ms (P95: {metrics['latencia_p95_ms']} ms)")
print(f"Rendimiento de procesamiento: {metrics['throughput_fps']} cuadros/segundo")
print(f"Confianza en aciertos: {metrics['confianza_aciertos']}% | En errores: {metrics['confianza_errores']}%")
print(f"Margen de certidumbre (Delta P): {metrics['margen_certidumbre']}%")
print('\nMétricas discriminadas por clase:')
print(f"{'Clase':30s} | {'Precisión':9s} | {'Recall':8s} | {'F1-Score':8s} | {'Muestras':8s}")
print('-' * 74)
for m in metrics['metricas_por_clase']:
    print(f"{m['name']:30s} | {m['precision']:8.2f}% | {m['recall']:7.2f}% | {m['f1']:7.2f}% | {m['muestras']:8d}")

cm = np.array(metrics['matriz_confusion'])
n_cls = len(classes)
names = [c['name'] for c in classes]

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(n_cls))
ax.set_yticks(range(n_cls))
ax.set_xticklabels(names, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(names, fontsize=8)
for i in range(n_cls):
    for j in range(n_cls):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=8,
                color='white' if cm[i, j] > cm.max() / 2 else 'black')
ax.set_xlabel('Clase predicha')
ax.set_ylabel('Clase real')
ax.set_title('Matriz de confusión en prueba (1,280 muestras)')
plt.tight_layout()
plt.show()

## Inspección visual de representaciones intermedias

Para analizar cómo la red transforma la información de entrada a lo largo de las capas, se extrae el promedio de activación de canales en tres etapas clave:
1. **Convolución inicial ($64 \times 64$):** Detección de trazos elementales, bordes y orientación.
2. **Transición 1 ($32 \times 32$):** Consolidación de uniones, bucles y grosor tras el primer bloque denso.
3. **Mapas profundos finales ($16 \times 16$):** Activaciones de alta abstracción morfológica previas al agrupamiento global.

In [ ]:
weights_path = os.path.join(base_dir, 'models', 'densenet_musical.pth')
checkpoint = torch.load(weights_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

sample_tensor, sample_label = test_loader.dataset[0]
acts = model.get_intermediate_activations(sample_tensor.unsqueeze(0).to(device))

img_in = sample_tensor.squeeze().numpy()
act_conv = acts['capa_inicial'][0].mean(dim=0).cpu().numpy()
act_trans = acts['transicion_1'][0].mean(dim=0).cpu().numpy()
act_deep = acts['mapas_profundos'][0].mean(dim=0).cpu().numpy()

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
axes[0].imshow(img_in, cmap='gray')
axes[0].set_title(f"Entrada ({classes[sample_label]['name']})", fontsize=8)
axes[0].axis('off')

axes[1].imshow(act_conv, cmap='inferno')
axes[1].set_title(f"Convolución inicial\n({acts['capa_inicial'].shape[1]} can, 64x64)", fontsize=8)
axes[1].axis('off')

axes[2].imshow(act_trans, cmap='inferno')
axes[2].set_title(f"Transición 1\n({acts['transicion_1'].shape[1]} can, 32x32)", fontsize=8)
axes[2].axis('off')

axes[3].imshow(act_deep, cmap='inferno')
axes[3].set_title(f"Mapas profundos\n({acts['mapas_profundos'].shape[1]} can, 16x16)", fontsize=8)
axes[3].axis('off')

plt.tight_layout()
plt.show()

## Inferencia de prueba sobre muestras individuales

Evaluación de predicción individual con el módulo desacoplado `MusicalSymbolPredictor`.

In [ ]:
predictor = MusicalSymbolPredictor(model_path=weights_path, device=device)

for i in [10, 60, 110, 160]:
    sample_img, label = test_loader.dataset[i]
    img_np = (sample_img.squeeze().numpy() * 255).astype(np.uint8)
    res = predictor.predict(img_np, top_k=2, generate_cam=False)
    real_name = classes[label]['name']
    pred_name = res['prediction']['name']
    pct = res['prediction']['percentage']
    print(f'Muestra #{i:3d} | Real: {real_name:26s} | Predicción: {pred_name:26s} ({pct:.1f}%)')

## Lógica y métodos principales de la red neuronal

El repositorio implementa módulos desacoplados en Python para la inferencia, extracción de características y canalización de datos:

### 1. `MusicalSymbolPredictor` (`src.predict`)
- **`predict(preprocessed_64x64: np.ndarray, top_k=4, generate_cam=False, return_layers=False) -> dict`:**
  - **Entrada:** Arreglo `uint8` $(64, 64)$ en $[0, 255]$ normalizado morfológicamente.
  - **Salida:** Diccionario con predicción dominante, lista `top_k`, probabilidades Softmax completas y mapas opcionales en base64.
  - **Comportamiento abstracto:** Convierte a tensor flotante $(1, 1, 64, 64)$ en $[0.0, 1.0]$, ejecuta la propagación sin gradientes (`torch.no_grad()`) y clasifica por probabilidad máxima.
- **`preprocess_image_file(file_path)` / `preprocess_pil(pil_img)` / `preprocess_base64(b64_str)`:**
  - **Entrada:** Ruta a disco, objeto Pillow o cadena base64.
  - **Salida:** Arreglo `np.ndarray` $(64, 64)$ binarizado y centrado morfológicamente por centroide.

### 2. `DenseNetMusical` (`src.model`)
- **`forward(x: torch.Tensor) -> torch.Tensor`:**
  - **Entrada:** Tensor $(B, 1, 64, 64)$ en $[0.0, 1.0]$.
  - **Salida:** Tensor de logits $(B, 8)$ antes de Softmax.
- **`extract_features(x: torch.Tensor) -> torch.Tensor`:**
  - **Salida:** Tensor de características profundas $(B, 92, 16, 16)$ previo a Global Average Pooling.
- **`get_intermediate_activations(x: torch.Tensor) -> dict`:**
  - **Salida:** Diccionario con representaciones intermedias de la convolución inicial, primera transición y mapas profundos.
- **`compute_cam(x: torch.Tensor, class_idx=None) -> tuple`:**
  - **Salida:** Mapa de activación de clase CAM $(64, 64)$ ponderado por la capa lineal.

### 3. Canalización de datos (`src.dataset`)
- **`create_dataloaders(data_file_path=None, batch_size=32) -> tuple`:**
  - **Salida:** `(train_loader, val_loader, test_loader, splits)`. Partición reproducible 70% entrenamiento, 10% validación y 20% prueba.
- **`load_classes_metadata(config_path=None) -> list`:**
  - **Salida:** Lista con especificaciones de las 8 clases musicales soportadas.

## Observaciones técnicas sobre casos particulares

1. **Corcheas unidas por barra / Doble negra (`beams`):**
   - *Comportamiento previo:* En la base de datos pública original (Rebelo et al.), esta categoría únicamente contemplaba trazos de barras horizontales o inclinadas segmentadas (sin plicas ni cabezas de nota). El modelo base reconocía con 100% de acierto dibujos de 2 a 3 líneas aisladas, pero no reconocía figuras completas trazadas por usuarios.
   - *Curación y adaptación:* De todas las combinaciones compuestas en notación musical, se seleccionó para la demostración el caso arquetípico de las **corcheas unidas por barra** (denominadas comúnmente *doble negra* o *figuras unidas por beam con plicas*). Se complementó el conjunto de entrenamiento con 150 dibujos manuales por integrante de esta figura completa en el lienzo.
   - *Resultado actual:* La red reconoce tanto las barras aisladas como la figura completa de corcheas unidas (doble negra). Su exhaustividad es del 95.35% (164 de 172 aciertos en prueba), con una precisión ligeramente menor que las figuras elementales debido a la mayor complejidad geométrica de unir tres trazos y a la leve confusión con trazos inclinados del silencio de negra.

2. **Estabilidad alcanzada en el silencio de negra (`Silencio: silencio de negra`):**
   - *Varianza en datos existentes:* Las muestras originales exhibían alta dispersión estilística y caligráfica (zigzags tipográficos antiguos vs. trazos manuscritos angulares), dificultando severamente el reconocimiento ante dibujos en pantalla (recall de 83.58% en el modelo sin datos propios).
   - *Estabilización con datos propios:* La incorporación de los 150 dibujos capturados en el lienzo interactivo redujo drásticamente el error, alcanzando una exhaustividad perfecta del **100.0% (140/140 aciertos en prueba)** y una precisión del 96.55%.

## Conclusiones

1. **Impacto de la recolección de datos propios en el dominio de inferencia:** La inclusión de 1,200 trazos interactivos capturados en el mismo lienzo de prueba elevó la exactitud Top-1 del 96.17% al **98.91%** y la Top-2 al **99.69%**, reduciendo los errores en prueba de 49 a tan solo 14 muestras.
2. **Eficiencia arquitectónica de DenseNet:** La concatenación directa de mapas permite alcanzar un desempeño superior al 98.9% con únicamente 80,520 parámetros entrenables, superando a arquitecturas convolucionales estándar de mayor complejidad.
3. **Factibilidad para inferencia en tiempo real:** Una latencia media de 14.57 ms (68.6 cuadros por segundo) en CPU ordinaria garantiza una respuesta interactiva imperceptible para el usuario.
4. **Calibración y confianza probabilística:** El margen de certidumbre de 28.26% entre aciertos (98.15%) y equivocaciones (69.90%) constata que el clasificador no produce sobreconfianza espuria ante trazos ambiguos.